# AfriAware — 01: Data Exploration

This notebook loads the available datasets and explores:
- Label distributions
- Text length distributions  
- Sample examples from each class
- Amharic character coverage

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.load_datasets import load_afrihate_amharic, load_ranlp_amharic
from src.data.preprocess import normalize_amharic

plt.style.use('seaborn-v0_8-whitegrid')
print('Setup complete')

## 1. Load datasets

In [ ]:
df_afrihate = load_afrihate_amharic()
df_ranlp    = load_ranlp_amharic()

## 2. Label distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

datasets = [
    ('AfriHate (amh)',  df_afrihate),
    ('RANLP (amh)',     df_ranlp),
]

for ax, (name, df) in zip(axes, datasets):
    if df is None or df.empty:
        ax.set_title(f'{name} — not loaded')
        continue
    col = 'label' if 'label' in df.columns else df.columns[-1]
    counts = df[col].value_counts()
    counts.plot(kind='bar', ax=ax, color=['#E8593C','#3B8BD4','#1D9E75'][:len(counts)])
    ax.set_title(name)
    ax.set_xlabel('')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)

plt.suptitle('Label distributions across datasets', y=1.02)
plt.tight_layout()
plt.show()

## 3. Text length analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (name, df) in zip(axes, [('AfriHate', df_afrihate), ('RANLP', df_ranlp)]):
    if df is None or df.empty:
        continue
    text_col = 'text' if 'text' in df.columns else 'tweet'
    lengths  = df[text_col].dropna().str.len()
    ax.hist(lengths, bins=40, color='#3B8BD4', alpha=0.8, edgecolor='white')
    ax.axvline(lengths.median(), color='#E8593C', linestyle='--', label=f'Median: {lengths.median():.0f}')
    ax.set_title(f'{name} — text length (chars)')
    ax.set_xlabel('Characters')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Sample examples per class

In [ ]:
print('=== AfriHate — sample by class ===')
if df_afrihate is not None and not df_afrihate.empty:
    text_col = 'text' if 'text' in df_afrihate.columns else 'tweet'
    for label in df_afrihate['label'].unique():
        sample = df_afrihate[df_afrihate['label'] == label][text_col].dropna().sample(2, random_state=42)
        print(f'\n[{label.upper()}]')
        for s in sample:
            print(f'  {s[:120]}')

## 5. Normalization preview

In [ ]:
samples = [
    'ሰላም @user ይህ ጥሩ ነው! 😊 https://t.co/abc123',
    '#ኢትዮጵያ ሀገራችን ናት!!!',
    'ጥሩ   ጽሑፍ    ነው',
]

for s in samples:
    print(f'BEFORE: {s}')
    print(f'AFTER : {normalize_amharic(s)}')
    print()